Pipeline clusters donnees historiques repertoire Sirene

Chargement et tirage stratifie des donnees entreprises depuis le jeu de donnees open data de data gouv
-tirage de 500 000 entreprises suivant la stratification: nombre de périodes * code NAF
-répartition des 500 000 en 10 échantillons représentatifs

In [ ]:
import pandas as pd
import numpy as np
import duckdb
import optuna
import hdbscan
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
from src.chargement_tirage_donnees import chargement_tirage

# DuckDB connexion
con = duckdb.connect(':memory:')
liste_echantillons = chargement_tirage(con)
con.close()

Exemple: chargement de l'échantillon 1 (~50 000 observations)

In [ ]:
features_ech1=liste_echantillons[0]
#verification des formats
features_ech1.dtypes

In [ ]:
#verification valeurs manquantes comptages
print("\nCount total NaN in the DataFrame:\n", features_ech1.isnull().sum())

Préparation des features avant la recherche de clusters (standardisation et PCA)
Récupération des poids de tirage pour les calculs de robustesse et significativité des clusters

In [ ]:
from src.preparation_donnees import preparation

X = preparation(features_ech1)

In [ ]:
weights=pd.DataFrame(features_ech1["poids_tirage"])

-Recherche des meilleurs hyperparamètres pour former des clusters d'entreprises par la méthode HDBSCAN

-Recherche d'optimisation des hyperparamètres min_cluster_size et min_samples suivant trois scores:

*au moins 2 clusters trouves

*score d'évaluation de la qualité du clustering (DBCV score) > 0.1

*minimisation des coeffcients de variation moyens des features au sein des clusters (maximum de l'opposé dans optuna qui ne gère que les max pas les min)

*stabilité-reproductibilité des clusters par rééchantillonnage avec remplacement (bootstrap)

-Optimisation effectuée via une fonction multi-objectifs (utilisation d'Optuna) sur 25 essais



In [ ]:
from src.robustesse_significativite_clusters import evaluate_significance,evaluate_stability_fast

# fonction multi-objectifs Optuna
def objective(trial):
    #si aucun des 25 essais n'est valide, revoir les intervalles des hyperparamètres infra
    min_cluster_size = trial.suggest_int("min_cluster_size", 1500,  16500) #intervalle de taille du cluster entre 1% et 33% de la taille de l'échantillon
    min_samples = trial.suggest_int("min_samples", 5, 100)
    
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        gen_min_span_tree=True,
        prediction_data=False, 
        core_dist_n_jobs=-1,
    )
    labels = clusterer.fit_predict(X)

    dbcv_score = clusterer.relative_validity_
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

    if np.isnan(dbcv_score) or dbcv_score < 0.1 or n_clusters < 2:
        return 0.0, -10.0

    
    stability_score = evaluate_stability_fast(
        X, weights, min_cluster_size, min_samples, labels, n_bootstraps=3 #3 bootstraps pour garder des temps de calcul acceptables
    )
    significance_score = evaluate_significance(X, weights, labels)

    return stability_score, significance_score



In [ ]:
# Lancement de l'optimisation
study = optuna.create_study(directions=["maximize", "maximize"])
study.optimize(objective, n_trials=25) #25 essais pour 50 000 oberservations ~ temps de calcul acceptable

# Sélection automatique du meilleur compromis entre stabilité et significativité
print("\n--- Meilleur compromis Stabilité vs Significativité ---")

best_trials = study.best_trials
best_trial = None
min_distance = float("inf")

# Objectifs théoriques parfaits : Stabilité = 1.0, Significativité (-CV) = 0.0
target_stability = 1.0
target_significance = 0.0

for trial in best_trials:
    # garde fou numéro 1 : Vérifier si le trial a des valeurs valides et complètes
    if trial.values is None or any(v is None for v in trial.values):
        continue

    stab, signif = trial.values

    # Calcul de la distance euclidienne par rapport au point "parfait"
    distance = np.sqrt(
        (target_stability - stab) ** 2 + (target_significance - signif) ** 2
    )

    if distance < min_distance:
        min_distance = distance
        best_trial = trial

# garde fou numéro 2 : Si aucun essai valide n'a été trouvé
if best_trial is None:
    print("Aucun essai valide n'a été trouvé. Essayer de modifier les intervalles des hyperparamètres proposés et/ou le nombre d'essais.")
else:
    print(f"Meilleur compromis trouvé (Trial {best_trial.number}) :")
    print(f" - Paramètres : {best_trial.params}")
    print(f" - Stabilité : {best_trial.values[0]:.4f}")
    print(f" - Significativité : {best_trial.values[1]:.4f}")

-Application de la recherche de clusters via la méthode HDBSCAN à partir des 2 hyper parametres optimises mentionnes supra

In [ ]:
df=features_ech1.drop(columns=["siren","poids_tirage","numero_echantillon"])

#jeu d'hyperparamètres optimisés
best_params = {
    "min_cluster_size": 1527,
    "min_samples":61,
}

#recherche de clusters via la méthode HDBSCAN
cluster = hdbscan.HDBSCAN(**best_params, gen_min_span_tree=True)

df["Cluster"] = cluster.fit_predict(X)

-Calcul des zscores (écart par features entre le cluster et l'échantillon qui le contient)

In [ ]:
from src.zscores import zscores

z_scores_ech1=zscores(df,weights)
z_scores_ech1.index=[f"Cluster_ech1_{i+1}" for i in range(len(z_scores_ech1))]



In [ ]:
z_scores_ech1

-Le calcul de la matrice des zscores est répété pour les 10 échantillons
-Une distance (similarité cosinus) est calculée entre les zscores de chaque échantillon pour détecter des faits saillants

In [ ]:
#dataframe de l'ensemble des zscores

echantillons = [
    z_scores_ech1, z_scores_ech2, z_scores_ech3, z_scores_ech4, z_scores_ech5,
    z_scores_ech6, z_scores_ech7, z_scores_ech8, z_scores_ech9, z_scores_ech10
]

z_scores = pd.concat(echantillons)

clusters = pd.concat([pd.Series(ech.index) for ech in echantillons])

In [ ]:
# Calcul de la similarité cosinus
# cosine_similarity renvoie une matrice symétrique (n_clusters x n_clusters)
matrix_similarity = cosine_similarity(z_scores)

# Conversion en DataFrame pour plus de lisibilité
df_similarity = pd.DataFrame(matrix_similarity,index=clusters, columns=clusters)